# Retail ETL Pipeline

This notebook contains the complete ETL pipeline for the Omnichannel Retail Data Warehouse lab.

**Tasks covered:**
1. Pipeline Configuration & Extract
2. Transform & Data Quality
3. Star Schema & Load
4. Idempotency & Incremental Loading
5. Orchestration & KPI Summary


## Task 1 - Pipeline Configuration & Extract


In [72]:
import pandas as pd
import sqlite3
import datetime
import logging
from dataclasses import dataclass
from typing import List, Tuple
import os

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class PipelineConfig:
    input_path: str
    db_path: str
    quarantine_path: str
    batches: List[str]

config = PipelineConfig(
    input_path="Python_Data_Pipeline_Lab_Dataset (1).xlsx",
    db_path="retail_dw.db",
    quarantine_path="quarantine.csv",
    batches=["orders_batch_1", "orders_batch_2", "orders_batch_3"]
)
print("PipelineConfig created:", config)


PipelineConfig created: PipelineConfig(input_path='Python_Data_Pipeline_Lab_Dataset (1).xlsx', db_path='retail_dw.db', quarantine_path='quarantine.csv', batches=['orders_batch_1', 'orders_batch_2', 'orders_batch_3'])


## Task 3 - Star Schema & Database Setup


In [73]:
def setup_database(db_path: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id  TEXT UNIQUE NOT NULL,
        customer_name TEXT,
        province     TEXT,
        segment      TEXT
    )''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_product (
        product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
        product_id   TEXT UNIQUE NOT NULL,
        product_name TEXT,
        category     TEXT
    )''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_date (
        date_key  INTEGER PRIMARY KEY,
        full_date TEXT UNIQUE NOT NULL,
        day       INTEGER,
        month     INTEGER,
        quarter   INTEGER,
        year      INTEGER
    )''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_sales (
        order_id       TEXT PRIMARY KEY,
        date_key       INTEGER NOT NULL,
        customer_key   INTEGER NOT NULL,
        product_key    INTEGER NOT NULL,
        quantity       INTEGER NOT NULL,
        unit_price     REAL    NOT NULL,
        discount_pct   REAL    NOT NULL,
        gross_amount   REAL    NOT NULL,
        net_amount     REAL    NOT NULL,
        payment_method TEXT,
        sales_channel  TEXT,
        FOREIGN KEY (date_key)     REFERENCES dim_date(date_key),
        FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key),
        FOREIGN KEY (product_key)  REFERENCES dim_product(product_key)
    )''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS pipeline_run_log (
        run_id          INTEGER PRIMARY KEY AUTOINCREMENT,
        batch_name      TEXT,
        started_at      TEXT,
        ended_at        TEXT,
        rows_read       INTEGER,
        rows_loaded     INTEGER,
        rows_rejected   INTEGER,
        rows_duplicated INTEGER,
        rows_skipped    INTEGER,
        status          TEXT
    )''')

    conn.commit()
    conn.close()
    logger.info("Database setup complete.")

setup_database(config.db_path)


2026-08-18 13:56:08,273 - INFO - Database setup complete.


## Task 1 (cont.) - Extract Function


In [74]:
def extract_data(config: PipelineConfig, batch_name: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    start = datetime.datetime.now()
    logger.info(f"[Extract] Starting for {batch_name}")

    try:
        customers_df = pd.read_excel(config.input_path, sheet_name='customers')
        products_df  = pd.read_excel(config.input_path, sheet_name='products')
        orders_df    = pd.read_excel(config.input_path, sheet_name=batch_name)

        elapsed = (datetime.datetime.now() - start).total_seconds()
        logger.info(f"[Extract] {batch_name}: {len(orders_df)} rows read in {elapsed:.2f}s")
        return customers_df, products_df, orders_df

    except Exception as e:
        logger.error(f"[Extract] Error reading {batch_name}: {e}")
        raise


## Task 2 - Transform & Data Quality


In [75]:
def transform_and_validate(
    orders_df: pd.DataFrame,
    customers_df: pd.DataFrame,
    products_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, int]:
    logger.info("[Transform] Starting transformation and validation")

    df = orders_df.copy()
    rows_before_dedup = len(df)

    # 1. Deduplicate by order_id (keep latest updated_at)
    df['updated_at'] = pd.to_datetime(df['updated_at'], errors='coerce')
    df = df.sort_values('updated_at').drop_duplicates('order_id', keep='last')
    rows_duplicated = rows_before_dedup - len(df)

    # 2. Safe type conversions
    df['quantity']       = pd.to_numeric(df['quantity'],       errors='coerce')
    df['unit_price']     = pd.to_numeric(df['unit_price'],     errors='coerce')
    df['discount_pct']   = pd.to_numeric(df['discount_pct'],   errors='coerce')
    df['order_datetime'] = pd.to_datetime(df['order_datetime'], errors='coerce')

    # 3. Normalize categorical columns
    payment_map = {
        'CREDIT CARD': 'CREDIT_CARD', 'CREDIT_CARD': 'CREDIT_CARD', 'CREDITCARD': 'CREDIT_CARD',
        'CASH': 'CASH', 'TRANSFER': 'TRANSFER', 'DEBIT': 'DEBIT',
        'QRCODE': 'QR_CODE', 'QR CODE': 'QR_CODE', 'QR_CODE': 'QR_CODE',
    }
    channel_map = {
        'ONLINE': 'ONLINE', 'IN-STORE': 'IN_STORE', 'IN_STORE': 'IN_STORE', 'INSTORE': 'IN_STORE',
        'MARKETPLACE': 'MARKETPLACE', 'LINE': 'LINE',
    }
    df['payment_method'] = df['payment_method'].astype(str).str.strip().str.upper().map(payment_map).fillna(df['payment_method'].astype(str).str.strip().str.upper())
    df['sales_channel']  = df['sales_channel'].astype(str).str.strip().str.upper().map(channel_map).fillna(df['sales_channel'].astype(str).str.strip().str.upper())

    # 4. Data Quality Rules
    df['reason_code'] = None

    mask = df['order_datetime'].isnull()
    df.loc[mask & df['reason_code'].isnull(), 'reason_code'] = 'INVALID_DATE'

    mask = (df['quantity'].isnull()) | (df['quantity'] <= 0)
    df.loc[mask & df['reason_code'].isnull(), 'reason_code'] = 'INVALID_QUANTITY'

    mask = (df['unit_price'].isnull()) | (df['unit_price'] <= 0)
    df.loc[mask & df['reason_code'].isnull(), 'reason_code'] = 'INVALID_PRICE'

    mask = (df['discount_pct'].isnull()) | (df['discount_pct'] < 0) | (df['discount_pct'] > 100)
    df.loc[mask & df['reason_code'].isnull(), 'reason_code'] = 'INVALID_DISCOUNT'

    valid_customers = set(customers_df['customer_id'].unique())
    mask = ~df['customer_id'].isin(valid_customers)
    df.loc[mask & df['reason_code'].isnull(), 'reason_code'] = 'CUSTOMER_NOT_FOUND'

    valid_products = set(products_df['product_id'].unique())
    mask = ~df['product_id'].isin(valid_products)
    df.loc[mask & df['reason_code'].isnull(), 'reason_code'] = 'PRODUCT_NOT_FOUND'

    # 5. Split clean vs quarantine
    clean_df      = df[df['reason_code'].isnull()].copy()
    quarantine_df = df[df['reason_code'].notnull()].copy()

    # 6. Derived columns
    clean_df['gross_amount'] = clean_df['quantity'] * clean_df['unit_price']
    clean_df['net_amount']   = clean_df['gross_amount'] * (1 - clean_df['discount_pct'] / 100.0)

    logger.info(f"[Transform] Done: {len(clean_df)} clean, {len(quarantine_df)} quarantined, {rows_duplicated} duplicates removed")
    return clean_df, quarantine_df, rows_duplicated


## Task 3 (cont.) - Load Function


In [76]:
def load_data(
    clean_df: pd.DataFrame,
    quarantine_df: pd.DataFrame,
    customers_df: pd.DataFrame,
    products_df: pd.DataFrame,
    config: PipelineConfig
) -> Tuple[int, int]:
    logger.info("[Load] Starting load to Data Warehouse")

    conn = sqlite3.connect(config.db_path)
    cursor = conn.cursor()
    rows_loaded  = 0
    rows_skipped = 0

    try:
        for _, row in customers_df.iterrows():
            cursor.execute('''
                INSERT OR IGNORE INTO dim_customer (customer_id, customer_name, province, segment)
                VALUES (?, ?, ?, ?)
            ''', (row['customer_id'], row['customer_name'], row['province'], row['segment']))

        for _, row in products_df.iterrows():
            cursor.execute('''
                INSERT OR IGNORE INTO dim_product (product_id, product_name, category)
                VALUES (?, ?, ?)
            ''', (row['product_id'], row['product_name'], row['category']))

        if not clean_df.empty:
            dates = clean_df['order_datetime'].dt.date.unique()
            for d in dates:
                date_key = int(d.strftime('%Y%m%d'))
                cursor.execute('''
                    INSERT OR IGNORE INTO dim_date (date_key, full_date, day, month, quarter, year)
                    VALUES (?, ?, ?, ?, ?, ?)
                ''', (date_key, str(d), d.day, d.month, (d.month - 1) // 3 + 1, d.year))

        conn.commit()

        cust_map = pd.read_sql("SELECT customer_id, customer_key FROM dim_customer", conn) \
                     .set_index('customer_id')['customer_key'].to_dict()
        prod_map = pd.read_sql("SELECT product_id, product_key FROM dim_product", conn) \
                     .set_index('product_id')['product_key'].to_dict()

        for _, row in clean_df.iterrows():
            date_key = int(row['order_datetime'].strftime('%Y%m%d'))
            c_key = cust_map.get(row['customer_id'])
            p_key = prod_map.get(row['product_id'])

            cursor.execute('''
                INSERT OR IGNORE INTO fact_sales
                (order_id, date_key, customer_key, product_key,
                 quantity, unit_price, discount_pct,
                 gross_amount, net_amount,
                 payment_method, sales_channel)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                row['order_id'], date_key, c_key, p_key,
                int(row['quantity']), float(row['unit_price']),
                float(row['discount_pct']),
                float(row['gross_amount']), float(row['net_amount']),
                row['payment_method'], row['sales_channel']
            ))

            if cursor.rowcount > 0:
                rows_loaded += 1
            else:
                rows_skipped += 1

        conn.commit()

        if not quarantine_df.empty:
            file_exists = os.path.isfile(config.quarantine_path)
            quarantine_df.to_csv(config.quarantine_path, mode='a', header=not file_exists, index=False)

        logger.info(f"[Load] Loaded {rows_loaded} new rows, skipped {rows_skipped} existing rows")

    except Exception as e:
        conn.rollback()
        logger.error(f"[Load] Error: {e}")
        raise
    finally:
        conn.close()

    return rows_loaded, rows_skipped


## Task 4 & 5 - Orchestration, Idempotency & KPI

**Formula:** `rows_read = rows_duplicated + rows_rejected + rows_loaded + rows_skipped`


In [77]:
def run_pipeline(config: PipelineConfig, batch_name: str):
    start_time = datetime.datetime.now()
    logger.info(f"{'='*60}")
    logger.info(f"Pipeline started for {batch_name}")
    logger.info(f"{'='*60}")

    status          = "SUCCESS"
    rows_read       = 0
    rows_loaded     = 0
    rows_rejected   = 0
    rows_duplicated = 0
    rows_skipped    = 0

    try:
        customers_df, products_df, orders_df = extract_data(config, batch_name)
        rows_read = len(orders_df)

        clean_df, quarantine_df, rows_duplicated = transform_and_validate(
            orders_df, customers_df, products_df
        )
        rows_rejected = len(quarantine_df)

        rows_loaded, rows_skipped = load_data(
            clean_df, quarantine_df, customers_df, products_df, config
        )

    except Exception as e:
        status = "FAILED: " + str(e)
        logger.error(status)

    finally:
        end_time = datetime.datetime.now()

        conn = sqlite3.connect(config.db_path)
        conn.execute('''
            INSERT INTO pipeline_run_log
            (batch_name, started_at, ended_at, rows_read, rows_loaded, rows_rejected, rows_duplicated, rows_skipped, status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (batch_name, str(start_time), str(end_time),
              rows_read, rows_loaded, rows_rejected, rows_duplicated, rows_skipped, status))
        conn.commit()
        conn.close()

        accounted = rows_loaded + rows_rejected + rows_duplicated + rows_skipped
        print("[" + batch_name + "] Read: " + str(rows_read)
              + " | Loaded: " + str(rows_loaded)
              + " | Rejected: " + str(rows_rejected)
              + " | Duplicated: " + str(rows_duplicated)
              + " | Skipped: " + str(rows_skipped)
              + " | Accounted: " + str(accounted)
              + " | Status: " + status)


## Execute Pipeline - 4 Runs


In [78]:
# Remove old outputs to start fresh
if os.path.exists(config.db_path):
    os.remove(config.db_path)
if os.path.exists(config.quarantine_path):
    os.remove(config.quarantine_path)

setup_database(config.db_path)

print("=" * 80)
print("Run 1: orders_batch_1")
print("=" * 80)
run_pipeline(config, "orders_batch_1")

print()
print("=" * 80)
print("Run 2: orders_batch_1 (Idempotency Test - should NOT load new rows)")
print("=" * 80)
run_pipeline(config, "orders_batch_1")

print()
print("=" * 80)
print("Run 3: orders_batch_2")
print("=" * 80)
run_pipeline(config, "orders_batch_2")

print()
print("=" * 80)
print("Run 4: orders_batch_3")
print("=" * 80)
run_pipeline(config, "orders_batch_3")


2026-08-18 13:56:08,310 - INFO - Database setup complete.
2026-08-18 13:56:08,310 - INFO - ============================================================
2026-08-18 13:56:08,311 - INFO - Pipeline started for orders_batch_1
2026-08-18 13:56:08,311 - INFO - ============================================================
2026-08-18 13:56:08,311 - INFO - [Extract] Starting for orders_batch_1
2026-08-18 13:56:08,439 - INFO - [Extract] orders_batch_1: 420 rows read in 0.13s
2026-08-18 13:56:08,439 - INFO - [Transform] Starting transformation and validation
2026-08-18 13:56:08,444 - INFO - [Transform] Done: 386 clean, 34 quarantined, 0 duplicates removed
2026-08-18 13:56:08,444 - INFO - [Load] Starting load to Data Warehouse
2026-08-18 13:56:08,458 - INFO - [Load] Loaded 386 new rows, skipped 0 existing rows
2026-08-18 13:56:08,459 - INFO - ============================================================
2026-08-18 13:56:08,459 - INFO - Pipeline started for orders_batch_1
2026-08-18 13:56:08,459 - INF

Run 1: orders_batch_1
[orders_batch_1] Read: 420 | Loaded: 386 | Rejected: 34 | Duplicated: 0 | Skipped: 0 | Accounted: 420 | Status: SUCCESS

Run 2: orders_batch_1 (Idempotency Test - should NOT load new rows)


2026-08-18 13:56:08,566 - INFO - [Extract] orders_batch_1: 420 rows read in 0.11s
2026-08-18 13:56:08,566 - INFO - [Transform] Starting transformation and validation
2026-08-18 13:56:08,570 - INFO - [Transform] Done: 386 clean, 34 quarantined, 0 duplicates removed
2026-08-18 13:56:08,571 - INFO - [Load] Starting load to Data Warehouse
2026-08-18 13:56:08,583 - INFO - [Load] Loaded 0 new rows, skipped 386 existing rows
2026-08-18 13:56:08,584 - INFO - ============================================================
2026-08-18 13:56:08,584 - INFO - Pipeline started for orders_batch_2
2026-08-18 13:56:08,584 - INFO - ============================================================
2026-08-18 13:56:08,585 - INFO - [Extract] Starting for orders_batch_2
2026-08-18 13:56:08,694 - INFO - [Extract] orders_batch_2: 424 rows read in 0.11s
2026-08-18 13:56:08,694 - INFO - [Transform] Starting transformation and validation
2026-08-18 13:56:08,699 - INFO - [Transform] Done: 383 clean, 38 quarantined, 3 dupl

[orders_batch_1] Read: 420 | Loaded: 0 | Rejected: 34 | Duplicated: 0 | Skipped: 386 | Accounted: 420 | Status: SUCCESS

Run 3: orders_batch_2
[orders_batch_2] Read: 424 | Loaded: 382 | Rejected: 38 | Duplicated: 3 | Skipped: 1 | Accounted: 424 | Status: SUCCESS

Run 4: orders_batch_3


2026-08-18 13:56:08,823 - INFO - [Extract] orders_batch_3: 424 rows read in 0.11s
2026-08-18 13:56:08,824 - INFO - [Transform] Starting transformation and validation
2026-08-18 13:56:08,828 - INFO - [Transform] Done: 383 clean, 38 quarantined, 3 duplicates removed
2026-08-18 13:56:08,828 - INFO - [Load] Starting load to Data Warehouse
2026-08-18 13:56:08,842 - INFO - [Load] Loaded 382 new rows, skipped 1 existing rows


[orders_batch_3] Read: 424 | Loaded: 382 | Rejected: 38 | Duplicated: 3 | Skipped: 1 | Accounted: 424 | Status: SUCCESS


## Results & KPI Summary


In [79]:
conn = sqlite3.connect(config.db_path)

print("Pipeline Run Log:")
print("-" * 120)
run_log = pd.read_sql("SELECT * FROM pipeline_run_log", conn)
display(run_log)

print()
fact_count = pd.read_sql("SELECT COUNT(*) as cnt FROM fact_sales", conn).iloc[0]['cnt']
cust_count = pd.read_sql("SELECT COUNT(*) as cnt FROM dim_customer", conn).iloc[0]['cnt']
prod_count = pd.read_sql("SELECT COUNT(*) as cnt FROM dim_product", conn).iloc[0]['cnt']
date_count = pd.read_sql("SELECT COUNT(*) as cnt FROM dim_date", conn).iloc[0]['cnt']
total_net  = pd.read_sql("SELECT COALESCE(SUM(net_amount), 0) as total FROM fact_sales", conn).iloc[0]['total']

print("Fact Sales count: " + str(fact_count))
print("Dim Customer count: " + str(cust_count))
print("Dim Product count: " + str(prod_count))
print("Dim Date count: " + str(date_count))
print("Total Net Sales: " + "{:,.2f}".format(total_net))

conn.close()


Pipeline Run Log:
------------------------------------------------------------------------------------------------------------------------


,run_id,batch_name,started_at,ended_at,rows_read,rows_loaded,rows_rejected,rows_duplicated,rows_skipped,status
0,1,orders_batch_1,2026-08-18 13:56:08.310896,2026-08-18 13:56:08.458494,420,386,34,0,0,SUCCESS
1,2,orders_batch_1,2026-08-18 13:56:08.459163,2026-08-18 13:56:08.583991,420,0,34,0,386,SUCCESS
2,3,orders_batch_2,2026-08-18 13:56:08.584692,2026-08-18 13:56:08.713928,424,382,38,3,1,SUCCESS
3,4,orders_batch_3,2026-08-18 13:56:08.714700,2026-08-18 13:56:08.842764,424,382,38,3,1,SUCCESS



Fact Sales count: 1150
Dim Customer count: 180
Dim Product count: 48
Dim Date count: 169
Total Net Sales: 2,813,971.88


## Acceptance Tests


In [80]:
conn = sqlite3.connect(config.db_path)

tests_passed = 0
tests_total  = 7

# Test 1
batches_run = set(pd.read_sql("SELECT DISTINCT batch_name FROM pipeline_run_log WHERE status='SUCCESS'", conn)['batch_name'])
t1 = {'orders_batch_1','orders_batch_2','orders_batch_3'}.issubset(batches_run)
tests_passed += t1
print("Test 1 - Pipeline ran all 3 batches: " + ("PASS" if t1 else "FAIL"))

# Test 2
dup_oid = pd.read_sql("SELECT order_id, COUNT(*) c FROM fact_sales GROUP BY order_id HAVING c>1", conn)
t2 = len(dup_oid) == 0
tests_passed += t2
print("Test 2 - order_id unique in fact_sales: " + ("PASS" if t2 else "FAIL") + " (" + str(len(dup_oid)) + " duplicates)")

# Test 3
orphans = {}
for dim, key in [('dim_customer','customer_key'), ('dim_product','product_key'), ('dim_date','date_key')]:
    n = pd.read_sql("SELECT COUNT(*) c FROM fact_sales f LEFT JOIN " + dim + " d ON f." + key + "=d." + key + " WHERE d." + key + " IS NULL", conn).iloc[0]['c']
    orphans[dim] = int(n)
t3 = all(v == 0 for v in orphans.values())
tests_passed += t3
print("Test 3 - FK integrity (orphans: " + str(orphans) + "): " + ("PASS" if t3 else "FAIL"))

# Test 4
neg = pd.read_sql('''
    SELECT SUM(CASE WHEN quantity<0 THEN 1 ELSE 0 END) nq,
           SUM(CASE WHEN unit_price<0 THEN 1 ELSE 0 END) np,
           SUM(CASE WHEN net_amount<0 THEN 1 ELSE 0 END) nn
    FROM fact_sales
''', conn).iloc[0]
nq = int(neg['nq']) if neg['nq'] is not None else 0
np_val = int(neg['np']) if neg['np'] is not None else 0
nn = int(neg['nn']) if neg['nn'] is not None else 0
t4 = (nq == 0) and (np_val == 0) and (nn == 0)
tests_passed += t4
print("Test 4 - No negative qty/price/net: " + ("PASS" if t4 else "FAIL"))

# Test 5
idem = pd.read_sql("SELECT rows_loaded FROM pipeline_run_log WHERE batch_name='orders_batch_1' ORDER BY run_id", conn)
t5 = len(idem) >= 2 and int(idem.iloc[1]['rows_loaded']) == 0
tests_passed += t5
print("Test 5 - Idempotency (batch_1 run2 loaded " + str(int(idem.iloc[1]['rows_loaded'])) + "): " + ("PASS" if t5 else "FAIL"))

# Test 6
if os.path.isfile(config.quarantine_path):
    q = pd.read_csv(config.quarantine_path)
    missing_reason = int(q['reason_code'].isnull().sum())
    t6 = missing_reason == 0
    tests_passed += t6
    print("Test 6 - All quarantine have reason_code: " + ("PASS" if t6 else "FAIL") + " (" + str(missing_reason) + " missing)")
    print("         Reason breakdown: " + str(q['reason_code'].value_counts().to_dict()))
else:
    t6 = False
    print("Test 6 - FAIL: quarantine.csv not found")

# Test 7
log = pd.read_sql("SELECT * FROM pipeline_run_log", conn)
all_match = True
for _, r in log.iterrows():
    accounted = int(r['rows_loaded']) + int(r['rows_rejected']) + int(r['rows_duplicated']) + int(r['rows_skipped'])
    read_val = int(r['rows_read'])
    match_str = "OK" if accounted == read_val else "MISMATCH"
    if match_str == "MISMATCH":
        all_match = False
    print("  Run " + str(int(r['run_id'])) + " [" + r['batch_name'] + "]: read=" + str(read_val) + " = loaded(" + str(int(r['rows_loaded'])) + ") + rejected(" + str(int(r['rows_rejected'])) + ") + dup(" + str(int(r['rows_duplicated'])) + ") + skipped(" + str(int(r['rows_skipped'])) + ") = " + str(accounted) + "  [" + match_str + "]")
t7 = all_match
tests_passed += t7
print("Test 7 - rows_read = loaded+rejected+dup+skipped: " + ("PASS" if t7 else "FAIL"))

conn.close()

print()
print("Result: " + str(tests_passed) + "/" + str(tests_total) + " tests passed")


Test 1 - Pipeline ran all 3 batches: PASS
Test 2 - order_id unique in fact_sales: PASS (0 duplicates)
Test 3 - FK integrity (orphans: {'dim_customer': 0, 'dim_product': 0, 'dim_date': 0}): PASS
Test 4 - No negative qty/price/net: PASS
Test 5 - Idempotency (batch_1 run2 loaded 0): PASS
Test 6 - All quarantine have reason_code: PASS (0 missing)
         Reason breakdown: {'INVALID_QUANTITY': 32, 'CUSTOMER_NOT_FOUND': 30, 'INVALID_DATE': 28, 'INVALID_PRICE': 24, 'INVALID_DISCOUNT': 16, 'PRODUCT_NOT_FOUND': 14}
  Run 1 [orders_batch_1]: read=420 = loaded(386) + rejected(34) + dup(0) + skipped(0) = 420  [OK]
  Run 2 [orders_batch_1]: read=420 = loaded(0) + rejected(34) + dup(0) + skipped(386) = 420  [OK]
  Run 3 [orders_batch_2]: read=424 = loaded(382) + rejected(38) + dup(3) + skipped(1) = 424  [OK]
  Run 4 [orders_batch_3]: read=424 = loaded(382) + rejected(38) + dup(3) + skipped(1) = 424  [OK]
Test 7 - rows_read = loaded+rejected+dup+skipped: PASS

Result: 7/7 tests passed


## Export pipeline_run_log.csv


In [81]:
conn = sqlite3.connect(config.db_path)
pd.read_sql("SELECT * FROM pipeline_run_log", conn).to_csv("pipeline_run_log.csv", index=False)
print("Exported pipeline_run_log.csv")
conn.close()


Exported pipeline_run_log.csv
